# ⚓ VesselWatch — Maritime Anomaly Detection

> **Author:** Jenish Patoliya  
> **Live Dashboard:** https://vesselwatch-vmzc2tetunqfqmv7wgblw7.streamlit.app/

---

## 📌 Problem Statement
Coast guards cannot manually monitor thousands of ships daily.
Criminals turn off GPS mid-ocean, transfer illegal cargo between
ships, then reappear looking normal — called a **dark voyage**.
Rule-based systems fail because criminals learn the rules.
VesselWatch uses ML to learn normal behavior and flag deviations.

---

## 📋 Pipeline Overview
```
Raw AIS Data (5.6M rows, 7 days)
        ↓  Section 1: Setup
        ↓  Section 2: Data Loading
        ↓  Section 3: Data Cleaning
        ↓  Section 4: Feature Engineering (15 features)
        ↓  Section 5: ML Models (IF + DBSCAN + LSTM)
        ↓  Section 6: Hyperparameter Tuning
        ↓  Section 7: SHAP Explainability
        ↓  Section 8: Validation
        ↓  Section 9: Save & Deploy
```

---

## 📊 Final Results
| Metric | Value |
|--------|-------|
| Vessels Analyzed | 16,937 |
| Flagged Vessels | 1,245 |
| Precision | 64.34% |
| Recall | 23.89% |
| F1 Score | 34.84% |
| **ROC-AUC** | **0.8076** |

---
## SECTION 1 — SETUP & LIBRARIES
Mounting Google Drive and importing all required libraries.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest
from sklearn.cluster import DBSCAN
from sklearn.metrics import (
    precision_score, recall_score,
    f1_score, roc_auc_score, roc_curve
)
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, LSTM, Dense,
    RepeatVector, TimeDistributed
)
import shap
import matplotlib.pyplot as plt
import gc
import json
import warnings
warnings.filterwarnings('ignore')

print('✅ All libraries imported')
print('TensorFlow:', tf.__version__)
print('Scikit-learn:', __import__('sklearn').__version__)

---
## SECTION 2 — DATA LOADING

**Dataset:** NOAA AIS Vessel Tracking Data  
**Coverage:** US Coastal Waters, January 11–17 2023  
**Volume:** ~5.6 million vessel pings across 7 consecutive days  
**Why 7 days?** Behavioral fingerprinting requires multiple days
to build a reliable baseline per vessel.

In [ ]:
RAW_PATH = '/content/drive/MyDrive/VesselWatch/data/raw/'

FILES = [
    'AIS_2023_01_11.csv',
    'AIS_2023_01_12.csv',
    'AIS_2023_01_13.csv',
    'AIS_2023_01_14.csv',
    'AIS_2023_01_15.csv',
    'AIS_2023_01_16.csv',
    'AIS_2023_01_17.csv',
]

USE_COLS = [
    'MMSI','BaseDateTime','LAT','LON',
    'SOG','COG','Heading','VesselName',
    'VesselType','Length'
]

dfs = []
for file in FILES:
    print(f'Loading {file}...')
    temp = pd.read_csv(
        RAW_PATH + file,
        usecols=USE_COLS,
        nrows=800000  # 800k rows per day for RAM efficiency
    )
    temp['date'] = file.split('_')[3].replace('.csv','')
    dfs.append(temp)

df = pd.concat(dfs, ignore_index=True)
del dfs
gc.collect()

print(f'\n✅ Data loaded successfully')
print(f'Total rows:    {len(df):,}')
print(f'Unique ships:  {df["MMSI"].nunique():,}')
print(f'Date range:    {df["BaseDateTime"].min()} to {df["BaseDateTime"].max()}')

---
## SECTION 3 — DATA CLEANING

Raw AIS data contains significant noise:
- **Fake MMSIs** — test transponders with sequential IDs
- **Impossible speeds** — sensor malfunctions showing >50 knots
- **Invalid headings** — value 511 means no GPS fix
- **Missing vessel names** — replaced with MMSI number
- **Duplicate pings** — same ship, same timestamp

In [ ]:
VESSEL_TYPE_MAP = {
    30:'Fishing', 31:'Towing', 32:'Towing',
    33:'Dredging', 34:'Diving', 35:'Military',
    36:'Sailing', 37:'Pleasure', 51:'SAR',
    52:'Tug', 60:'Passenger', 61:'Passenger',
    62:'Passenger', 63:'Passenger', 69:'Passenger',
    70:'Cargo', 71:'Cargo', 72:'Cargo',
    73:'Cargo', 79:'Cargo', 80:'Tanker',
    81:'Tanker', 82:'Tanker', 83:'Tanker',
    89:'Tanker'
}

print(f'Rows before cleaning: {len(df):,}')

# Fix data types
df['BaseDateTime'] = pd.to_datetime(df['BaseDateTime'])

# Remove fake/invalid MMSIs
df = df[(df['MMSI'] >= 200000000) & (df['MMSI'] <= 999999999)]

# Remove impossible speeds
df = df[(df['SOG'] >= 0) & (df['SOG'] <= 50)]

# Fix invalid heading (511 = no GPS fix)
df['Heading'] = df['Heading'].replace(511, np.nan)

# Fill missing values
df['VesselName'] = df['VesselName'].fillna(df['MMSI'].astype(str))
df['VesselType'] = df['VesselType'].fillna(0)
df['Length'] = df['Length'].fillna(0)

# Remove duplicates and sort
df = df.drop_duplicates(subset=['MMSI','BaseDateTime'])
df = df.sort_values(['MMSI','BaseDateTime']).reset_index(drop=True)

# Map vessel type codes to readable labels
df['VesselTypeLabel'] = df['VesselType'].map(
    VESSEL_TYPE_MAP
).fillna('Other')

# Save cleaned data
df.to_parquet(
    '/content/drive/MyDrive/VesselWatch/data/processed/ais_cleaned_v2.parquet',
    index=False
)

print(f'Rows after cleaning:  {len(df):,}')
print(f'Rows removed:         {5600000-len(df):,}')
print(f'\nVessel type distribution:')
print(df['VesselTypeLabel'].value_counts().head(8))
print('\n✅ Cleaned data saved to Drive')

---
## SECTION 4 — FEATURE ENGINEERING

**Why feature engineering?**  
Raw GPS pings cannot be fed directly to ML models.
We transform 5.6M pings into 15 behavioral features
per vessel — collapsing the data from 5.6M rows to
16,937 rows (one per vessel).

| # | Feature | Description |
|---|---------|-------------|
| 1 | speed_mean | Average speed over 7 days |
| 2 | speed_std | Speed standard deviation |
| 3 | speed_min | Minimum recorded speed |
| 4 | speed_max | Maximum recorded speed |
| 5 | speed_variance | Erratic speed changes |
| 6 | total_distance_km | Total distance traveled |
| 7 | total_time_hrs | Total active hours |
| 8 | loitering_score | Distance vs time ratio |
| 9 | max_gap_hrs | Longest AIS signal gap |
| 10 | total_gaps | Number of AIS gaps |
| 11 | gap_flag | AIS gap occurred yes/no |
| 12 | position_jump_km | Distance jumped after gap |
| 13 | dist_from_port_km | Distance from nearest port |
| 14 | behavioral_score | ★ Deviation from own baseline |
| 15 | speed_consistency | Daily speed consistency |

**★ Behavioral Fingerprinting** — compares each vessel
against its OWN 7-day history, not the fleet average.
A ship that normally moves at 8 knots but suddenly
appears at 14 knots is flagged — even if 14 knots
is normal for other ships.

In [ ]:
from math import radians, sin, cos, sqrt, atan2

def haversine(lat1, lon1, lat2, lon2):
    """Calculate distance between two GPS coordinates in km"""
    R = 6371
    lat1,lon1,lat2,lon2 = map(radians,[lat1,lon1,lat2,lon2])
    dlat = lat2-lat1
    dlon = lon2-lon1
    a = sin(dlat/2)**2 + cos(lat1)*cos(lat2)*sin(dlon/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1-a))

# ── Feature 1-5: Speed Statistics ─────────────────────
print('Computing speed statistics...')
speed_features = df.groupby('MMSI')['SOG'].agg(
    speed_mean='mean',
    speed_std='std',
    speed_min='min',
    speed_max='max',
    speed_variance=lambda x: x.var()
).reset_index().fillna(0)

# ── Feature 6-8: Loitering Score ──────────────────────
print('Computing loitering scores...')
def calc_loitering(group):
    """Low distance + high time = high loitering score"""
    if len(group) < 2:
        return pd.Series({
            'total_distance_km':0,
            'total_time_hrs':0,
            'loitering_score':0
        })
    group = group.sort_values('BaseDateTime')
    lats = group['LAT'].values
    lons = group['LON'].values
    dist = sum(
        haversine(lats[i],lons[i],lats[i+1],lons[i+1])
        for i in range(len(lats)-1)
    )
    hrs = (group['BaseDateTime'].max() -
           group['BaseDateTime'].min()
           ).total_seconds()/3600
    score = 1/(1+(dist/(hrs+0.001))) if hrs>0 else 0
    return pd.Series({
        'total_distance_km': round(dist,3),
        'total_time_hrs': round(hrs,3),
        'loitering_score': round(score,4)
    })

loitering_features = df.groupby('MMSI').apply(
    calc_loitering
).reset_index()

# ── Feature 9-12: AIS Gap Detection ───────────────────
print('Detecting AIS gaps...')
def detect_gaps(group):
    """Find signal disappearances > 2 hours in open ocean"""
    if len(group) < 2:
        return pd.Series({
            'max_gap_hrs':0,'total_gaps':0,
            'gap_flag':0,'position_jump_km':0
        })
    group = group.sort_values('BaseDateTime')
    diffs = group['BaseDateTime'].diff().dt.total_seconds()/3600
    gaps = diffs[diffs > 2]
    jump = 0
    if len(gaps) > 0:
        idx = diffs.idxmax()
        iloc = group.index.get_loc(idx)
        if iloc > 0:
            jump = haversine(
                group['LAT'].iloc[iloc-1],
                group['LON'].iloc[iloc-1],
                group['LAT'].iloc[iloc],
                group['LON'].iloc[iloc]
            )
    return pd.Series({
        'max_gap_hrs': round(diffs.max(),3),
        'total_gaps': len(gaps),
        'gap_flag': 1 if len(gaps)>0 else 0,
        'position_jump_km': round(jump,3)
    })

gap_features = df.groupby('MMSI').apply(
    detect_gaps
).reset_index()

# ── Feature 13: Distance From Port ────────────────────
print('Computing port distances...')
ports = pd.read_csv(
    '/content/drive/MyDrive/VesselWatch/data/raw/world_ports.csv'
)[['Main Port Name','Latitude','Longitude']].dropna()

last_pos = df.groupby('MMSI').agg(
    last_lat=('LAT','last'),
    last_lon=('LON','last')
).reset_index()

def nearest_port(lat, lon):
    d = np.sqrt(
        (ports['Latitude']-lat)**2 +
        (ports['Longitude']-lon)**2
    )
    return round(d.min()*111, 2)

last_pos['dist_from_port_km'] = last_pos.apply(
    lambda r: nearest_port(r['last_lat'],r['last_lon']),
    axis=1
)

# ── Feature 14-15: Behavioral Fingerprinting ──────────
print('Computing behavioral fingerprints...')
daily = df.groupby(
    ['MMSI', df['BaseDateTime'].dt.date]
).agg(
    daily_speed=('SOG','mean'),
    daily_pings=('SOG','count')
).reset_index()

baseline = daily.groupby('MMSI').agg(
    baseline_speed=('daily_speed','mean'),
    speed_consistency=('daily_speed','std')
).reset_index().fillna(0)

# Behavioral score = deviation from own baseline
baseline['behavioral_score'] = (
    baseline['speed_consistency'] /
    (baseline['baseline_speed']+0.001)
).round(4)

# ── Combine All Features ──────────────────────────────
print('\nCombining all features...')
features = speed_features.copy()
features = features.merge(
    loitering_features[['MMSI','total_distance_km',
    'total_time_hrs','loitering_score']],
    on='MMSI', how='left'
)
features = features.merge(
    gap_features[['MMSI','max_gap_hrs','total_gaps',
    'gap_flag','position_jump_km']],
    on='MMSI', how='left'
)
features = features.merge(
    last_pos[['MMSI','last_lat','last_lon',
    'dist_from_port_km']],
    on='MMSI', how='left'
)
features = features.merge(
    baseline[['MMSI','behavioral_score',
    'speed_consistency']],
    on='MMSI', how='left'
)
features = features.merge(
    df.groupby('MMSI')['VesselTypeLabel'].first().reset_index(),
    on='MMSI', how='left'
)
features = features.merge(
    df.groupby('MMSI')['VesselName'].first().reset_index(),
    on='MMSI', how='left'
)
features = features.fillna(0)

features.to_parquet(
    '/content/drive/MyDrive/VesselWatch/data/processed/vessel_features_v2.parquet',
    index=False
)
print(f'✅ Feature engineering complete')
print(f'Shape: {features.shape}')
print(f'From {len(df):,} pings → {len(features):,} vessel profiles')

---
## SECTION 5 — ML MODELS

Three complementary models, each catching different anomaly types:

| Model | Purpose | Final Weight |
|-------|---------|-------------|
| **Isolation Forest** | Overall behavioral outliers | 70% |
| **DBSCAN** | Rendezvous detection in open ocean | 30% |
| **LSTM Autoencoder** | Trajectory sequence anomalies | Implemented* |

*LSTM is fully trained but excluded from final ensemble.
See Section 6 for explanation.

In [ ]:
ML_COLS = [
    'speed_mean','speed_std','speed_min','speed_max',
    'speed_variance','total_distance_km','total_time_hrs',
    'loitering_score','max_gap_hrs','total_gaps','gap_flag',
    'position_jump_km','dist_from_port_km',
    'behavioral_score','speed_consistency'
]

X = features[ML_COLS].replace([np.inf,-np.inf],0).fillna(0)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ── MODEL 1: Isolation Forest ─────────────────────────
# Contamination=0.1 chosen after tuning (see Section 6)
print('Training Isolation Forest (contamination=0.1)...')
iso = IsolationForest(
    n_estimators=200,
    contamination=0.1,  # tuned from 0.05
    random_state=42,
    n_jobs=-1
)
iso.fit(X_scaled)
iso_pred = iso.predict(X_scaled)
iso_scores = iso.decision_function(X_scaled)
iso_risk = 1-(iso_scores-iso_scores.min())/(iso_scores.max()-iso_scores.min())
features['iso_risk_score'] = iso_risk
features['iso_flag'] = (iso_pred==-1).astype(int)
print(f'✅ Isolation Forest: {features["iso_flag"].sum()} flagged')

# ── MODEL 2: DBSCAN Rendezvous Detection ──────────────
print('Running DBSCAN rendezvous detection...')
moving = features[
    (features['MMSI']>=200000000) &
    (features['speed_mean']>=0.5)
].copy()
coords = np.radians(moving[['last_lat','last_lon']].values)
db = DBSCAN(eps=0.009, min_samples=2,
            metric='haversine', n_jobs=-1)
clusters = db.fit_predict(coords)
moving['cluster_id'] = clusters
moving['rendezvous_flag'] = (
    (moving['cluster_id']!=-1) &
    (moving['dist_from_port_km']>100)
).astype(int)
features['rendezvous_flag'] = 0
features['rendezvous_risk'] = 0.0
features.loc[moving.index,'rendezvous_flag'] = moving['rendezvous_flag'].values
features.loc[moving.index,'rendezvous_risk'] = moving['rendezvous_flag'].astype(float).values
print(f'✅ DBSCAN: {features["rendezvous_flag"].sum()} rendezvous flags')

# ── MODEL 3: LSTM Autoencoder ─────────────────────────
print('Building and training LSTM Autoencoder...')
X_lstm = X_scaled.reshape(X_scaled.shape[0],1,X_scaled.shape[1])
inp = Input(shape=(1, X_scaled.shape[1]))
enc = LSTM(64, activation='relu', return_sequences=False)(inp)
rep = RepeatVector(1)(enc)
dec = LSTM(64, activation='relu', return_sequences=True)(rep)
out = TimeDistributed(Dense(X_scaled.shape[1]))(dec)
autoencoder = Model(inp, out)
autoencoder.compile(optimizer='adam', loss='mse')
normal_mask = features['iso_risk_score'] < 0.5
autoencoder.fit(
    X_lstm[normal_mask], X_lstm[normal_mask],
    epochs=50, batch_size=256,
    validation_split=0.1,
    shuffle=True, verbose=0
)
X_recon = autoencoder.predict(X_lstm, verbose=0)
recon_err = np.mean(np.power(X_lstm-X_recon,2), axis=(1,2))
lstm_risk = (recon_err-recon_err.min())/(recon_err.max()-recon_err.min())
features['lstm_risk_score'] = lstm_risk
features['lstm_flag'] = (recon_err > np.percentile(recon_err,95)).astype(int)
print(f'✅ LSTM Autoencoder trained | Flagged: {features["lstm_flag"].sum()}')
print(f'   LSTM mean reconstruction error: {recon_err.mean():.6f}')

# ── COMBINE SCORES ────────────────────────────────────
# Note: LSTM excluded from ensemble (see Section 6)
features['final_risk_score'] = (
    (features['iso_risk_score'] * 0.70) +
    (features['rendezvous_risk'] * 0.30)
).round(4)
features['final_flag'] = (
    features['final_risk_score'] >= 0.5
).astype(int)

print(f'\n✅ Final Risk Scores Combined')
print(f'High risk (>0.7):  {len(features[features["final_risk_score"]>0.7])}')
print(f'Medium (0.5-0.7):  {len(features[(features["final_risk_score"]>=0.5)&(features["final_risk_score"]<0.7)])}')
print(f'Total flagged:     {features["final_flag"].sum()}')
print(f'\nTop 5 suspicious vessels:')
print(features.nlargest(5,'final_risk_score')[
    ['VesselName','VesselTypeLabel','final_risk_score',
     'max_gap_hrs','position_jump_km']
].to_string())

---
## SECTION 6 — HYPERPARAMETER TUNING

### Why We Tuned
Initial results showed recall of only 13.54%.
Systematic tuning improved it to 23.89%.

### Isolation Forest Contamination Tuning

| Contamination | Precision | Recall | F1 | AUC |
|--------------|-----------|--------|----|-----------|
| 0.01 | 70.59% | 3.58% | 6.81% | 0.8060 |
| 0.02 | 71.09% | 7.19% | 13.06% | 0.8060 |
| 0.03 | 69.94% | 10.62% | 18.44% | 0.8060 |
| 0.05 | 68.00% | 17.18% | 27.43% | 0.8060 |
| 0.08 | 60.89% | 24.60% | 35.05% | 0.8060 |
| **0.10** | **58.21%** | **29.41%** | **39.07%** | **0.8060** |

**Chosen: contamination=0.10** — best F1 score

### Why LSTM Was Excluded From Ensemble

After training, LSTM reconstruction error mean = **0.0004** (near zero).
This means LSTM scores were not differentiating between
normal and suspicious vessels.

**Root cause:** 7-day window gives limited trajectory length
per vessel for meaningful sequence learning.

**Decision:** Exclude LSTM from ensemble score.
Keep implementation in notebook to demonstrate deep learning
capability. With 30+ days data, LSTM would contribute meaningfully.

### Before vs After Tuning

| Metric | Before | After |
|--------|--------|-------|
| Precision | 66.86% | 64.34% |
| **Recall** | 13.54% | **23.89%** |
| **F1 Score** | 22.52% | **34.84%** |
| **ROC-AUC** | 0.8060 | **0.8076** |
| Flagged | 679 | 1,245 |

In [ ]:
# Tuning experiment code
print('=== ISOLATION FOREST CONTAMINATION TUNING ===')

features['true_anomaly'] = (
    (features['max_gap_hrs'] > 24) |
    (features['position_jump_km'] > 200) |
    (features['dist_from_port_km'] > 300)
).astype(int)

results = []
for c in [0.01, 0.02, 0.03, 0.05, 0.08, 0.10]:
    iso_t = IsolationForest(
        n_estimators=200, contamination=c,
        random_state=42, n_jobs=-1
    )
    iso_t.fit(X_scaled)
    pred = iso_t.predict(X_scaled)
    scores = iso_t.decision_function(X_scaled)
    risk = 1-(scores-scores.min())/(scores.max()-scores.min())
    flag = (pred==-1).astype(int)
    p = precision_score(features['true_anomaly'], flag, zero_division=0)
    r = recall_score(features['true_anomaly'], flag, zero_division=0)
    f = f1_score(features['true_anomaly'], flag, zero_division=0)
    auc = roc_auc_score(features['true_anomaly'], risk)
    results.append({'contamination':c,'precision':p,'recall':r,'f1':f,'auc':auc})
    print(f'c={c} | P:{p:.2%} R:{r:.2%} F1:{f:.2%} AUC:{auc:.4f}')

print(f'\n✅ Best contamination: 0.10 (highest F1)')

---
## SECTION 7 — SHAP EXPLAINABILITY

**Why SHAP?**  
A risk score of 0.89 is useless without explanation.
A coast guard analyst cannot board a vessel because
an algorithm said 0.89 — they need specific, legally
defensible reasons.

SHAP (SHapley Additive exPlanations) tells exactly
which feature drove each flag and by how much.

**Interpretation:**
- **Negative SHAP value** → pushed toward anomaly
- **Positive SHAP value** → pushed toward normal

In [ ]:
print('Calculating SHAP values...')
explainer = shap.TreeExplainer(iso)
shap_values = explainer.shap_values(X_scaled)
print(f'✅ SHAP values calculated: {shap_values.shape}')

# Save SHAP for top 50 vessels
top50 = features.nlargest(50,'final_risk_score')
shap_df = pd.DataFrame(shap_values, columns=ML_COLS)
shap_df['MMSI'] = features['MMSI'].values
shap_df['VesselName'] = features['VesselName'].values
shap_df['final_risk_score'] = features['final_risk_score'].values
top_shap = shap_df.iloc[top50.index.tolist()].copy()
top_shap.to_parquet(
    '/content/drive/MyDrive/VesselWatch/data/processed/shap_explanations_v2.parquet',
    index=False
)

# Show explanation for most suspicious vessel
top = top_shap.iloc[0]
print(f'\n=== SHAP EXPLANATION ===')
print(f'Vessel: {top["VesselName"]}')
print(f'Risk:   {top["final_risk_score"]}')
print('\nFeature contributions (sorted by importance):')
vals = [(abs(top[c]),c,top[c]) for c in ML_COLS]
for _,col,val in sorted(vals,reverse=True):
    d = '↑ ANOMALY' if val<0 else '↓ NORMAL'
    bar = '█' * int(abs(val)*3)
    print(f'  {col:25s}: {val:+.4f}  {d}  {bar}')

---
## SECTION 8 — VALIDATION

**Validation approach:**  
No perfect labeled dataset exists for maritime crime.
We use behavioral pseudo-labels — vessels with extreme
AIS gaps, position jumps, or port distances are labeled
as true anomalies.

**Why conservative recall is acceptable:**  
In maritime surveillance, false positives waste coast guard
resources. A model that flags fewer but more certain cases
is operationally preferable.

In [ ]:
# Pseudo labels with behavioral thresholds
features['true_anomaly'] = (
    (features['max_gap_hrs'] > 24) |
    (features['position_jump_km'] > 200) |
    (features['dist_from_port_km'] > 300)
).astype(int)

p = precision_score(features['true_anomaly'], features['final_flag'], zero_division=0)
r = recall_score(features['true_anomaly'], features['final_flag'], zero_division=0)
f1_val = f1_score(features['true_anomaly'], features['final_flag'], zero_division=0)
auc_val = roc_auc_score(features['true_anomaly'], features['final_risk_score'])

high = features[features['final_risk_score']>0.7]
low  = features[features['final_risk_score']<0.3]

print('=== VALIDATION METRICS ===')
print(f'Precision:     {p:.2%}')
print(f'Recall:        {r:.2%}')
print(f'F1 Score:      {f1_val:.2%}')
print(f'ROC-AUC:       {auc_val:.4f}  ← key metric')
print(f'Flagged:       {features["final_flag"].sum()}')

print('\n=== BEHAVIORAL VALIDATION ===')
print(f'High risk avg AIS gap:    {high["max_gap_hrs"].mean():.1f} hrs')
print(f'Normal avg AIS gap:       {low["max_gap_hrs"].mean():.1f} hrs')
print(f'Ratio:                    {high["max_gap_hrs"].mean()/low["max_gap_hrs"].mean():.1f}x larger')
print(f'')
print(f'High risk avg jump:       {high["position_jump_km"].mean():.1f} km')
print(f'Normal avg jump:          {low["position_jump_km"].mean():.1f} km')
print(f'Ratio:                    {high["position_jump_km"].mean()/low["position_jump_km"].mean():.0f}x larger')

# ROC Curve
fpr, tpr, _ = roc_curve(features['true_anomaly'], features['final_risk_score'])
fig, ax = plt.subplots(figsize=(8,6))
fig.patch.set_facecolor('#111827')
ax.set_facecolor('#111827')
ax.plot(fpr, tpr, color='#00d4ff', linewidth=2,
        label=f'VesselWatch (AUC={auc_val:.4f})')
ax.plot([0,1],[0,1], color='#4a5568', linestyle='--',
        label='Random Classifier')
ax.fill_between(fpr, tpr, alpha=0.1, color='#00d4ff')
ax.set_xlabel('False Positive Rate', color='white')
ax.set_ylabel('True Positive Rate', color='white')
ax.set_title('VesselWatch ROC Curve', color='#00d4ff', fontsize=14)
ax.tick_params(colors='white')
ax.legend(facecolor='#1e2d45', labelcolor='white')
ax.grid(True, alpha=0.2, color='#1e2d45')
plt.tight_layout()
plt.savefig(
    '/content/drive/MyDrive/VesselWatch/data/processed/roc_curve.png',
    dpi=150, facecolor='#111827'
)
plt.show()
print('✅ ROC curve saved')

---
## SECTION 9 — SAVE & DEPLOY

Saving all outputs for dashboard deployment on Streamlit Cloud.

In [ ]:
# Save parquet files
features.to_parquet(
    '/content/drive/MyDrive/VesselWatch/data/processed/final_results_v2.parquet',
    index=False
)
top_shap.to_parquet(
    '/content/drive/MyDrive/VesselWatch/data/processed/shap_explanations_v2.parquet',
    index=False
)

# Save CSV for Streamlit Cloud deployment
features.to_csv(
    '/content/drive/MyDrive/VesselWatch/final_results.csv',
    index=False
)
top_shap.to_csv(
    '/content/drive/MyDrive/VesselWatch/shap_explanations.csv',
    index=False
)

# Save validation summary
summary = {
    'contamination': 0.1,
    'model_weights': 'IsolationForest=0.70, DBSCAN=0.30',
    'lstm_note': 'Trained but excluded — recon error mean=0.0004',
    'total_vessels': int(len(features)),
    'total_flagged': int(features['final_flag'].sum()),
    'high_risk': int(len(features[features['final_risk_score']>0.7])),
    'precision': round(float(p),4),
    'recall': round(float(r),4),
    'f1_score': round(float(f1_val),4),
    'roc_auc': round(float(auc_val),4),
    'dashboard_url': 'https://vesselwatch-vmzc2tetunqfqmv7wgblw7.streamlit.app/',
    'github_url': 'https://github.com/JenishPatoliya/VesselWatch'
}

with open(
    '/content/drive/MyDrive/VesselWatch/validation_summary.json','w'
) as f:
    json.dump(summary, f, indent=2)

print('✅ ALL FILES SAVED SUCCESSFULLY')
print('\n=== FINAL PROJECT SUMMARY ===')
print(f'Total vessels analyzed:  {len(features):,}')
print(f'Suspicious vessels:      {features["final_flag"].sum():,}')
print(f'Precision:               {p:.2%}')
print(f'Recall:                  {r:.2%}')
print(f'F1 Score:                {f1_val:.2%}')
print(f'ROC-AUC:                 {auc_val:.4f}')
print(f'\nDashboard: https://vesselwatch-vmzc2tetunqfqmv7wgblw7.streamlit.app/')
print(f'GitHub:    https://github.com/JenishPatoliya/VesselWatch')

---
## ⚡ QUICK LOAD — Skip Reprocessing

Run this single cell to load saved results instantly.
No need to rerun entire pipeline every session.

In [ ]:
# =============================================
# QUICK START — Run this after mounting Drive
# Loads everything in 30 seconds
# =============================================

features = pd.read_parquet(
    '/content/drive/MyDrive/VesselWatch/data/processed/final_results_v2.parquet'
)
shap_df = pd.read_parquet(
    '/content/drive/MyDrive/VesselWatch/data/processed/shap_explanations_v2.parquet'
)

print('✅ Project loaded successfully')
print(f'Vessels:  {len(features):,}')
print(f'Flagged:  {features["final_flag"].sum():,}')
print(f'High risk:{len(features[features["final_risk_score"]>0.7]):,}')
print(f'\nDashboard: https://vesselwatch-vmzc2tetunqfqmv7wgblw7.streamlit.app/')